In [1]:
import time
import pandas as pd
import numpy as np  # Import NumPy for numerical operations
from pypradie import Tensor, Sequential, Linear, ReLU
from pypradie.optim import SGD
from pypradie.nn import CrossEntropyLoss
from pypradie.utils.data import DataLoader, TensorDataset

## Step 1: Load and Prepare the MNIST Dataset from CSV
train_df = pd.read_csv('../../datasets/mnist/mnist_train.csv')
test_df = pd.read_csv('../../datasets/mnist/mnist_test.csv')

# For testing purposes, let's use a smaller subset of the dataset
train_df = train_df.head(1000)
test_df = test_df.head(100)

# Normalize pixel values to [0, 1] range
train_labels = train_df['label'].values
train_features = train_df.drop('label', axis=1).values / 255.0  # Normalize the pixel values
test_labels = test_df['label'].values
test_features = test_df.drop('label', axis=1).values / 255.0  # Normalize the pixel values

# Convert data to tensors using PyPradie
train_features_tensor = Tensor(train_features, dtype=np.float32)
train_labels_tensor = Tensor(train_labels, dtype=np.int64)
test_features_tensor = Tensor(test_features, dtype=np.float32)
test_labels_tensor = Tensor(test_labels, dtype=np.int64)

# Create TensorDataset and DataLoader with batch size of 1 (for simplicity)
batch_size = 1
train_dataset = TensorDataset(train_features_tensor, train_labels_tensor)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

## Step 2: Define a Simple Model (Sequential)
model = Sequential(
    Linear(784, 128),  # Input layer (28x28 pixels = 784)
    ReLU(),
    Linear(128, 64),
    ReLU(),
    Linear(64, 10),    # Output layer for 10 classes
)

## Step 3: Define Loss Function and Optimizer
criterion = CrossEntropyLoss()
optimizer = SGD(model.parameters(), lr=0.01)

## Step 4: Training Loop with Batch Size of 1
epochs = 3
correct_train = 0
total_train = 0

for epoch in range(epochs):
    start_time = time.time()
    for images, labels in train_loader:
        # Reset gradients
        optimizer.zero_grad()

        # Forward pass: Compute predictions
        outputs = model(images)

        # Compute loss
        loss = criterion(outputs, labels)
        
        # Backward pass: Compute gradients
        loss.backward()

        # Perform optimization step (update weights)
        optimizer.step()

        # Calculate train accuracy for this batch
        predicted = outputs.argmax(dim=1)  # Get the predicted class
        total_train += labels.size(0)  # Increment the total number of samples
        correct_train += (predicted == labels).sum().item()  # Check if prediction is correct

    # Print the training accuracy at the end of each epoch
    train_accuracy = 100 * correct_train / total_train
    epoch_time = time.time() - start_time
    print(f"Epoch {epoch + 1}/{epochs}, Time: {epoch_time:.2f} seconds, Train Accuracy: {train_accuracy:.2f}%")


Epoch 1/3, Time: 0.56 seconds, Train Accuracy: 100.00%
Epoch 2/3, Time: 0.58 seconds, Train Accuracy: 100.00%
Epoch 3/3, Time: 0.56 seconds, Train Accuracy: 100.00%
